---
# Chapter 4 — From Retrieval to Persistent Understanding

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 4: From Retrieval to Persistent Understanding |
| Central question | What do we gain by preserving AI-derived understanding as reusable state rather than reconstructing it from raw evidence independently on every query? |
| Main concepts | Persistent derived memory, GraphRAG, Entity resolution, Provenance mapping, Indexing cost |
| Implementation | graph_memory (Microsoft GraphRAG backend) |
| Experiment | ch4-20260919T205622Z (GraphRAG vs baseline) |
| Evidence status | Developmental verdict: Type C with Type B core |
| Depends on | Chapter 3 (baseline), Chapter 2 (instrument) |

---

## What this notebook demonstrates

This chapter tests whether **persistent derived interpretation** (GraphRAG) beats strong conventional RAG. The notebook:

1. **Loads the frozen GraphRAG index** over the Chapter 3 fixture corpus
2. **Inspects the derived graph**: 20 documents, 39 entities, 65 relationships, 56 claims, 6 communities
3. **Shows the frozen comparison results** across Basic, Local, Global query modes
4. **Demonstrates the central finding**: Strong RAG remains extremely competitive; graph does not produce general quality win; costs are large
5. **Shows persistent mistakes**: Entity resolution failures (typo nodes) that persist across queries

> **Evidence status**: Developmental verdict (Type C with Type B core). The frozen run `ch4-20260919T205622Z` exists. GraphRAG matches baseline on local decision questions, improves on relational/global questions (untested), keeps provenance mappable, costs substantially more.

## The chapter question

> **What do we gain by preserving AI-derived understanding as reusable state rather than reconstructing it from raw evidence independently on every query?**

The transition is not from dumb matching to smart comprehension (Chapter 3's reader already understands at query time). It is from **transient interpretation to persistent interpretation**:

> **RAG interprets history for this query. Persistent graph memory preserves some of that interpretation for future queries.**

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(4)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 4 Concepts")

## The three layers (critical distinction)

The chapter enforces three layers that fail differently:

| Layer | What it is | Epistemic status |
|-------|------------|------------------|
| **Layer 1 — Raw source** | What actually existed: session, commit, decision record, benchmark | Authoritative. Not the system's opinion. |
| **Layer 2 — Persistent derived interpretation** | What indexing inferred: entities, relationships, claims, communities | Reusable, versioned. Where extraction errors live permanently. |
| **Layer 3 — Query-relative use** | What matters for the current question | Computed per query. Must not be stored as permanent truth. |

> **Derived memory is a hypothesis about history, not a rewrite of history.**

Layer 2 records *derivation provenance* (this graph object came from those text units). It does **not** record *evidential support* (whether those sources license the eventual claim). Graph connectivity ≠ truth.

## Load the frozen GraphRAG run

The chapter's frozen comparison: `ch4-20260919T205622Z`

In [ ]:
import json

from notebooks.memory._support import load_frozen_run

run = load_frozen_run("ch4-20260919T205622Z")

print(f"Run ID: {run['run_id']}")
print(f"Conditions: {sorted(run.get('condition_details', {}).keys())}")

adj_path = REPO_ROOT / "experiments" / "benchmark" / "runs" / run["run_id"] / "adjudication.json"
adj = json.loads(adj_path.read_text(encoding="utf-8"))
print(f"\nAdjudication method: {adj['method'][:160]}...")
print("Adjudicated decision accuracy:", adj["table"])
print(f"\nVerdict: {adj['verdict']}")

## The frozen index statistics (from the chapter)

> **Book result.** The frozen index over the Chapter 3 fixture corpus (`ch3-fixture-v0.1`, corpus hash `04aac354`) contains:

- 20 documents
- 20 text units
- 39 entities
- 65 relationships
- 56 claims
- 6 communities
- 5 community reports (community 2 missing — summariser output failed schema validation)

Indexing ran 999 seconds wall time, 55 LLM responses, ~128,000 tokens using `ministral-3:8b` for extraction and `bge-m3` for embeddings.

In [ ]:
# Display the index stats as a table
index_stats = [
    {"Component": "Documents", "Count": 20},
    {"Component": "Text units", "Count": 20},
    {"Component": "Entities", "Count": 39},
    {"Component": "Relationships", "Count": 65},
    {"Component": "Claims", "Count": 56},
    {"Component": "Communities", "Count": 6},
    {"Component": "Community reports", "Count": 5},
    {"Component": "Indexing wall time", "Count": "999 s"},
    {"Component": "LLM responses", "Count": 55},
    {"Component": "Tokens (est.)", "Count": "~128,000"},
]
render_table(index_stats, "Frozen GraphRAG Index Statistics (Book Result)")

## Provenance mapping (Book Result)

> **Book result.** All 166 derived objects (39 entities, 65 relationships, 56 claims, 6 communities) map back to canonical source artifacts. Orphan rate is zero in every kind. Source mapping tells us where each derived object came from — this is **derivation provenance**, not evidential support.

## Persistent mistakes (Book Result)

> **Book result.** Persistent understanding creates reusable intelligence **and reusable mistakes**. The frozen index exhibits:

- `J. LINDQVIST` (degree 4) and `J. LINQVIST` (degree 3) — same person, split by typo
- `EVENT-STORE` and `EVENT STORE` — same system, two spellings, different declared kinds
- `A. NOVAK` and `A.NOVAK` — spacing-split pattern

These are **stored, versioned, queryable mistakes** inherited by every downstream consumer (association Ch5, routing Ch6, lineage Ch7) until rebuild.

In [ ]:
# Display persistent mistakes
mistakes = [
    {"Entity Pair": "J. LINDQVIST / J. LINQVIST", "Issue": "Typo split (one-letter extraction error)", "Relationships affected": 3, "Includes": "J. LINQVIST → REDIS (Session-040 proposal)"},
    {"Entity Pair": "EVENT-STORE / EVENT STORE", "Issue": "Spelling split (hyphen vs space)", "Relationships affected": "Different declared kinds", "Includes": "Both anchor migration graph"},
    {"Entity Pair": "A. NOVAK / A.NOVAK", "Issue": "Spacing split pattern", "Relationships affected": "Multiple", "Includes": "Participant in discussions"},
    {"Entity Pair": "Community 2", "Issue": "Missing report (summariser failed schema validation)", "Relationships affected": "Structural absence", "Includes": "Logged as indexing failure"},
]
render_table(mistakes, "Persistent Mistakes in Frozen Index (Book Result)")

## The frozen comparison results

Mechanical scores from the frozen artifacts (14 tasks from `ch4-tasks-v0.1`, 7 families, `llama3.1:8b` reader shared):

In [ ]:
# Results from the chapter text
comparison_results = [
    {"Condition": "Chapter 3 best", "Tasks": 14, "Source Recall": 1.000, "Decision Exactness": "0.889 (8/9)", "Mean Context Tokens": 580, "Mean Latency": "~7 s (reader-side)"},
    {"Condition": "Graph Basic", "Tasks": 14, "Source Recall": 0.819, "Decision Exactness": "0.889 (8/9)", "Mean Context Tokens": 1317, "Mean Latency": "51.5 s"},
    {"Condition": "Graph Local", "Tasks": 14, "Source Recall": 0.819, "Decision Exactness": "0.667 (6/9)", "Mean Context Tokens": 1250, "Mean Latency": "92.7 s"},
    {"Condition": "Graph Global", "Tasks": 8, "Source Recall": 0.812, "Decision Exactness": "0.800 (4/5)", "Mean Context Tokens": 1525, "Mean Latency": "196.1 s (max 585 s)"},
]
render_table(comparison_results, "Frozen Comparison: GraphRAG vs Baseline (Mechanical Scores)")

## Adjudicated rescoring (Chapter 4 analysis)

After rescuing cells whose answers contain the expected mechanism in paraphrase (alias list omissions) or near-verbatim word-order variants:

- **Best**: 8/9
- **Basic**: 9/9
- **Local**: 8/9
- **Global**: 4/5 (on its covered tasks)

The largest gap between any two full-coverage conditions is **one case in nine** — that is noise, not victory in any direction.

## Per-family detail

- **Temporal questions (3/3)**: Recall everywhere, all current/historical answers correct — solved by every condition (baseline reader handles supersession from raw passages)
- **Relational case (`q-rel-influences`)**: Baseline misses thinly; Basic and Local answer with explicit mechanism — *the run's one genuine point for stored structure composing across artifacts*
- **Synthesis case (`q-global-reliability`)**: Local misses — genuinely vaguer than baseline, names performance issues without contention/benchmark/incident specifics
- **Global mode's synthesis territory**: Untested by design (6 missing cells including both global-family and both relational-family tasks)

## What GraphRAG actually bought us (costs)

| Cost | Measured |
|------|----------|
| Index build | 999 s wall, 55 LLM responses, ~128k tokens for 20 docs |
| Query latency | Basic ~51s, Local ~93s, Global ~196s (max 585s) |
| Context | 2.2–2.6× baseline tokens for equal/indistinguishable quality |
| Storage | Full parquet index + embedding stores |
| Model calls/query | Uncounted by package API (recorded as unavailable) |

Against that price: **measured gains on direct QA are zero in general**. Adjudicated decision accuracy indistinguishable across full-coverage conditions.

## Where structure may still help (narrower claims)

1. **Relational composition**: One case (`q-rel-influences`) where stored relationships composed across 5 artifacts
2. **Corpus-wide synthesis**: Untested, not refuted — Global ran none of the synthesis questions
3. **Downstream consumption**: Chapters 5, 6, 7 genuinely consume the graph — *option value*, not empirical quality

> **Developmental verdict: Type C with Type B core.** On direct QA, strong RAG remains sufficient (Type B). Graph supplies persistent substrate that associative retrieval, selective routing, and evidence lineage consume (Type C).

## What this establishes

- **RAG already understands** — Chapter 3's reader does substantial interpretation at query time
- **Persistent ≠ better** — Stored interpretation adds cost, persistent errors, and no general quality win on local workload
- **Provenance is complete but distinct from support** — Derivation provenance (Layer 2) ≠ evidential support (Chapter 7)
- **Persistent mistakes are structurally different** — Silent, load-bearing, expensive to fix (rebuild required)
- **Mitigation is boundary, not prevention** — Raw sources stay canonical; fallback to strong retrieval; versioned rebuilds

## What this does NOT establish

- No general quality win for GraphRAG over strong RAG
- No cross-query consistency measurement (recorded as pending obligation)
- No tested synthesis family (Global's home territory untested)
- No relational family with ledger-derived scoring (recorded as obligation)
- Scorer normalisation defects (alias lists, citation format) need repair before rerun counts as publication-grade

## Try it yourself

The graph index is a parquet store. If you have the GraphRAG package installed, you can inspect entities, relationships, and communities directly. Key questions to explore:

- Which entities have the highest degree? (SQLITE degree 17, POSTGRESQL degree 8)
- What relationships connect the contention evidence to the decision?
- Which community report summarises the Redis rejection?
- What happens if you query the typo-split entities?

In [ ]:
# TRY IT YOURSELF: If you have graphrag installed, you can load the index
try:
    import graphrag
    print("graphrag package available")
except ImportError:
    print("graphrag not installed in this environment")
    print("The frozen index is stored as parquet files in the GraphRAG working directory.")
    print("To inspect: pip install graphrag and point to the index directory.")

## Where this leads next

The graph is static — relationships sit until a query arrives. But remembering does something else: a cue touches one memory, activation moves along relations, and the needed memory arrives three steps later by a route no similarity computation planned.

Chapter 5 asks: **How recall should move through the graph — and whether moving buys anything that lookup cannot.**

> **See this chapter in code:** [Open the companion Jupyter notebook](memory-chapter.ipynb)